- plan mode：collaboration mode
    - request_user_input: Request user input for one to three short questions and wait for the response. This tool is only available in Plan mode.
- emergent behavior
    - update_plan
        - Updates the task plan. Provide an optional explanation and a list of plan items, each with a step and status. At most one step can be in_progress at a time.
- understanding dynamics (of agentic loop): create / update(status) / replan
    - plan: steps / to-dos
- general tool (system) design of codex
    - general/unified
    - minimal/clean 

### plan mode

- `/plan`: switch to plan mode
    - role=developer message
        - https://github.com/openai/codex/blob/main/codex-rs/collaboration-mode-templates/templates/plan.md
    - `files/test_query_wire_request_6160.json`
- Plan mode 是一种协作模式，目标是“聊出一个可直接实施的规格”，最终产物是一个包在 `<proposed_plan>...</proposed_plan>` 里的 Markdown 计划（字符串，非文件）；它明确要求非变更式探索、尽量用 `request_user_input` 提问、并且“直到 developer message 显式结束前都处于 Plan mode”。
    - 这个最终计划在协议层对应 ThreadItem::Plan { id, text }，所以它是“线程项/历史项”，不是磁盘上的 markdown 文件。
    - Plan mode 的最终 plan 会以 TurnItem::Plan 的完成事件形式被保留，用于后续 replay / thread/read。
```
  Implement this plan?

› 1. Yes, implement this plan  Switch to Default and start coding.
  2. No, stay in Plan mode     Continue planning with the model.
```
- 它通过 collaboration preset（预设） 注入一整套行为约束，要求先探索、再澄清、再形成决策完备的计划，并尽量通过 request_user_input 结构化提问。

### update_plan

- update_plan 是执行期的 TODOs/checklist 工具（live progress model，一个对当前进度状态的内部表示），用来给用户显示“当前任务进度”，不是进入/退出 Plan mode 的开关，而且在 Plan mode 里会直接报错。
- 输入 schema 很简单：`{ explanation?: string, plan: [{ step: string, status: pending|in_progress|completed }] }`。
    - 输出几乎没有业务语义。handler 返回给模型的是 "Plan updated" 的 function-call output；code mode 结果是空对象 {}。源码注释甚至直说“有用的是输入，不是输出”。
    - `FunctionCallOutput("Plan updated")`
- base instructions
    - https://github.com/openai/codex/blob/main/codex-rs/protocol/src/prompts/base_instructions/default.md
        - Planning
        - Tool Guidelines
            - update_plan: pending/in_progress/completed
- 同一个 turn 内如何更新 plan
    - 不是“修改旧对象”，而是再次发一个新的 update_plan function call，并带上完整的新 plan 快照。

```json

{
    "instructions": "<session_meta.base_instructions.text>",
    "input": [
      {
        "type": "message",
        "role": "developer",
        "content": [
          {"type": "input_text", "text": "<permissions instructions>..."},
          {"type": "input_text", "text": "<collaboration_mode>Default...</collaboration_mode>"},
          {"type": "input_text", "text": "<apps_instructions>..."},
          {"type": "input_text", "text": "<skills_instructions>..."},
          {"type": "input_text", "text": "<plugins_instructions>..."}
        ]
      },
      {
        "type": "message",
        "role": "user",
        "content": [
          {"type": "input_text", "text": "# AGENTS.md instructions ..."},
          {"type": "input_text", "text": "<environment_context>..."}
        ]
      },
      {
        "type": "message",
        "role": "user",
        "content": [
          {"type": "input_text", "text": "REAL first User Query"}
        ]
      }
    ]
  }
  
  {
    "explanation": "Tracing how plan mode is modeled across prompts, tool specs, runtime orchestration, protocol, and local state so I can answer trigger conditions, update_plan
  semantics, and persistence behavior end-to-end.",
    "plan": [
      {"step": "Locate plan-related symbols and runtime mode switches across the repo", "status": "in_progress"},
      {"step": "Inspect tool definitions, wire/protocol structures, and prompt/context assembly", "status": "pending"},
      {"step": "Trace any local persistence or markdown generation paths for plan artifacts", "status": "pending"},
      {"step": "Synthesize design analysis covering triggers, lifecycle, and tradeoffs", "status": "pending"}
    ]
  }

  {
    "explanation": "Finished tracing plan mode presets, update_plan tool semantics, app-server notifications, plan-item streaming, and rollout persistence boundaries.",
    "plan": [
      {"step": "Locate plan-related symbols and runtime mode switches across the repo", "status": "completed"},
      {"step": "Inspect tool definitions, wire/protocol structures, and prompt/context assembly", "status": "completed"},
      {"step": "Trace any local persistence or markdown generation paths for plan artifacts", "status": "completed"},
      {"step": "Synthesize design analysis covering triggers, lifecycle, and tradeoffs", "status": "in_progress"}
    ]
  }
```

#### dynamics

> Sometimes, you may need to change plans in the middle of a task: call `update_plan` with the updated plan and make sure to provide an `explanation` of the rationale when doing so.

- 在 tool/runtime 这一层，update_plan 并没有“create”和“update”两个不同 API。
    - 所以从语义上说，它更像“提交当前计划快照”。
- 第一次 update_plan 时，会将第一个 step 置为 in_progress
- 完全可以更新整个 plan，不只是改 step 的 status。